# Hyperparameter tuning and model selection

Tune both baseline model families with stratified group cross-validation and select the final candidate using internal validation macro F1.

In [1]:
from pathlib import Path
import sys
import time

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import (
    build_logistic_regression_pipeline,
    build_svm_pipeline,
    get_logistic_regression_param_grid,
    get_svm_param_grid
)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\jadka\OneDrive\Documents\progressSoft_internship\phase-1-machine-learning-nlp\assignment


In [2]:
TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_clean.csv"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validation_clean.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Training shape: (50301, 4)
Validation shape: (12682, 4)


In [3]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

X_validation = validation_df["text"]
y_validation = validation_df["sentiment"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))
print("\nTraining label distribution:")
print(y_train.value_counts())

Training samples: 50301
Validation samples: 12682

Training label distribution:
sentiment
Negative      15533
Positive      13761
Neutral       12157
Irrelevant     8850
Name: count, dtype: int64


In [4]:
training_groups = (
    train_df["entity"].astype(str)
    + "::"
    + train_df["tweet_id"].astype(str)
)

cv_strategy = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

# Logistic Regression Tuning

In [5]:
logistic_param_grid = get_logistic_regression_param_grid()


In [6]:
logistic_search = GridSearchCV(
    estimator=build_logistic_regression_pipeline(),
    param_grid=logistic_param_grid,
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    refit=True,
)

In [7]:
start_time = time.perf_counter()

logistic_search.fit(X_train, y_train, groups=training_groups)

logistic_search_time = (
    time.perf_counter() - start_time
)

print(
    f"Logistic Regression search time: "
    f"{logistic_search_time:.2f} seconds"
)

print(
    "Best parameters:",
    logistic_search.best_params_,
)

print(
    "Best cross-validation macro F1:",
    logistic_search.best_score_,
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


Logistic Regression search time: 261.18 seconds
Best parameters: {'classifier__C': 0.5, 'classifier__class_weight': 'balanced'}
Best cross-validation macro F1: 0.5281862901518932


In [8]:
logistic_cv_results = pd.DataFrame(
    logistic_search.cv_results_
)

logistic_cv_summary = (
    logistic_cv_results[
        [
            "params",
            "mean_train_score",
            "std_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

display(logistic_cv_summary)

,params,mean_train_score,std_train_score,mean_test_score,std_test_score,rank_test_score,mean_fit_time
0,"{'classifier__C': 0.5, 'classifier__class_weig...",0.949314,0.000935,0.528186,0.004734,1,69.249621
1,"{'classifier__C': 1.0, 'classifier__class_weig...",0.977646,0.001192,0.525028,0.005867,2,69.447177
2,"{'classifier__C': 2.0, 'classifier__class_weig...",0.990399,0.000525,0.521251,0.004723,3,98.008319
3,"{'classifier__C': 0.1, 'classifier__class_weig...",0.809795,0.000385,0.519974,0.004920,4,41.451364
4,"{'classifier__C': 5.0, 'classifier__class_weig...",0.996512,0.000410,0.515662,0.003818,5,61.404134
5,"{'classifier__C': 1.0, 'classifier__class_weig...",0.975166,0.000320,0.513326,0.005168,6,49.261164
6,"{'classifier__C': 2.0, 'classifier__class_weig...",0.989477,0.000586,0.511883,0.003963,7,85.918803
7,"{'classifier__C': 5.0, 'classifier__class_weig...",0.996735,0.000269,0.509864,0.003386,8,88.833677
8,"{'classifier__C': 0.5, 'classifier__class_weig...",0.937989,0.000905,0.509113,0.006383,9,71.601866
9,"{'classifier__C': 0.1, 'classifier__class_weig...",0.661150,0.000853,0.465357,0.006906,10,51.789519


# Linear SVM Tuning

In [9]:
svm_param_grid = get_svm_param_grid()

svm_search = GridSearchCV(
    estimator=build_svm_pipeline(),
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    refit=True,
)

In [10]:
start_time = time.perf_counter()

svm_search.fit(X_train, y_train, groups=training_groups)

svm_search_time = (
    time.perf_counter() - start_time
)

print(
    f"SVM search time: "
    f"{svm_search_time:.2f} seconds"
)

print(
    "Best parameters:",
    svm_search.best_params_,
)

print(
    "Best cross-validation macro F1:",
    svm_search.best_score_,
)

Fitting 3 folds for each of 14 candidates, totalling 42 fits


SVM search time: 165.37 seconds
Best parameters: {'classifier__C': 0.05, 'classifier__class_weight': 'balanced'}
Best cross-validation macro F1: 0.5227079011332653


In [11]:
svm_cv_results = pd.DataFrame(
    svm_search.cv_results_
)

svm_cv_summary = (
    svm_cv_results[
        [
            "params",
            "mean_train_score",
            "std_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

display(svm_cv_summary)

,params,mean_train_score,std_train_score,mean_test_score,std_test_score,rank_test_score,mean_fit_time
0,"{'classifier__C': 0.05, 'classifier__class_wei...",0.927561,0.001268,0.522708,0.005708,1,18.815827
1,"{'classifier__C': 0.1, 'classifier__class_weig...",0.970238,0.000488,0.521864,0.005911,2,18.924628
2,"{'classifier__C': 0.5, 'classifier__class_weig...",0.995799,0.000193,0.510373,0.004410,3,20.404034
3,"{'classifier__C': 0.1, 'classifier__class_weig...",0.965362,0.000205,0.509855,0.005753,4,17.834977
4,"{'classifier__C': 0.5, 'classifier__class_weig...",0.995653,0.000170,0.505468,0.004136,5,20.486502
5,"{'classifier__C': 1.0, 'classifier__class_weig...",0.997571,0.000165,0.503561,0.004403,6,24.482602
6,"{'classifier__C': 0.01, 'classifier__class_wei...",0.764220,0.002210,0.501981,0.005426,7,18.796568
7,"{'classifier__C': 1.0, 'classifier__class_weig...",0.997542,0.000135,0.501023,0.003594,8,20.847825
8,"{'classifier__C': 0.05, 'classifier__class_wei...",0.906370,0.001454,0.499444,0.006319,9,18.829941
9,"{'classifier__C': 2.0, 'classifier__class_weig...",0.998328,0.000021,0.496079,0.004072,10,27.715108


In [12]:
def evaluate_model(model,X,y,model_name: str):

    start_time = time.perf_counter()

    predictions = model.predict(X)

    prediction_time = (
        time.perf_counter() - start_time
    )

    accuracy = accuracy_score(
        y,
        predictions,
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        y,
        predictions,
        average="macro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        y,
        predictions,
        average="weighted",
        zero_division=0,
    )

    metrics = pd.DataFrame(
        {
            "model": [model_name],
            "accuracy": [accuracy],
            "macro_precision": [
                macro_precision
            ],
            "macro_recall": [macro_recall],
            "macro_f1": [macro_f1],
            "weighted_precision": [
                weighted_precision
            ],
            "weighted_recall": [
                weighted_recall
            ],
            "weighted_f1": [weighted_f1],
            "prediction_time_seconds": [
                prediction_time
            ],
        }
    )

    predictions_df = pd.DataFrame(
        {
            "actual": y.to_numpy(),
            "predicted": predictions,
        }
    )

    return metrics, predictions_df

In [13]:
tuned_logistic_metrics, tuned_logistic_predictions = (
    evaluate_model(
        model=logistic_search.best_estimator_,
        X=X_validation,
        y=y_validation,
        model_name="Tuned Logistic Regression",
    )
)

display(tuned_logistic_metrics)

print(
    classification_report(
        y_validation,
        tuned_logistic_predictions["predicted"],
        zero_division=0,
    )
)

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,prediction_time_seconds
0,Tuned Logistic Regression,0.568207,0.546631,0.545637,0.545755,0.565561,0.568207,0.566465,1.800072


              precision    recall  f1-score   support

  Irrelevant       0.38      0.36      0.37      2228
    Negative       0.63      0.68      0.66      3898
     Neutral       0.55      0.55      0.55      3072
    Positive       0.62      0.59      0.61      3484

    accuracy                           0.57     12682
   macro avg       0.55      0.55      0.55     12682
weighted avg       0.57      0.57      0.57     12682



In [14]:
tuned_svm_metrics, tuned_svm_predictions = (
    evaluate_model(
        model=svm_search.best_estimator_,
        X=X_validation,
        y=y_validation,
        model_name="Tuned Linear SVM",
    )
)

display(tuned_svm_metrics)

print(
    classification_report(
        y_validation,
        tuned_svm_predictions["predicted"],
        zero_division=0,
    )
)

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,prediction_time_seconds
0,Tuned Linear SVM,0.573569,0.548091,0.544026,0.543211,0.563952,0.573569,0.566212,1.769445


              precision    recall  f1-score   support

  Irrelevant       0.41      0.31      0.35      2228
    Negative       0.62      0.72      0.66      3898
     Neutral       0.56      0.54      0.55      3072
    Positive       0.61      0.61      0.61      3484

    accuracy                           0.57     12682
   macro avg       0.55      0.54      0.54     12682
weighted avg       0.56      0.57      0.57     12682



In [15]:
baseline_comparison = pd.read_csv(
    PROJECT_ROOT / "reports/results/baseline_model_comparison.csv"
)

tuned_comparison = pd.concat([
    tuned_logistic_metrics.assign(
        training_or_search_time_seconds=logistic_search_time
    ),
    tuned_svm_metrics.assign(
        training_or_search_time_seconds=svm_search_time
    ),
])

model_comparison = pd.concat(
    [baseline_comparison, tuned_comparison],
    ignore_index=True,
).sort_values(by="macro_f1", ascending=False).reset_index(drop=True)

model_comparison

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,training_or_search_time_seconds,weighted_precision,weighted_recall,prediction_time_seconds
0,Tuned Logistic Regression,0.568207,0.546631,0.545637,0.545755,0.566465,261.180591,0.565561,0.568207,1.800072
1,Tuned Linear SVM,0.573569,0.548091,0.544026,0.543211,0.566212,165.368116,0.563952,0.573569,1.769445
2,Baseline Logistic Regression,0.573175,0.547069,0.539104,0.536343,0.561306,15.284238,NaN,NaN,NaN
3,Baseline Linear SVM,0.545340,0.518786,0.514926,0.513174,0.536523,9.582194,NaN,NaN,NaN


In [16]:
RESULTS_DIR = (
    PROJECT_ROOT
    / "reports"
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [17]:
logistic_cv_summary.to_csv(
    RESULTS_DIR
    / "logistic_regression_tuning_results.csv",
    index=False,
)

svm_cv_summary.to_csv(
    RESULTS_DIR
    / "svm_tuning_results.csv",
    index=False,
)


model_comparison.to_csv(
    RESULTS_DIR
    / "tuned_model_comparison.csv",
    index=False,
)

In [18]:
best_parameters = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "best_parameters": str(
                logistic_search.best_params_
            ),
            "best_cv_macro_f1": (
                logistic_search.best_score_
            ),
        },
        {
            "model": "Linear SVM",
            "best_parameters": str(
                svm_search.best_params_
            ),
            "best_cv_macro_f1": (
                svm_search.best_score_
            ),
        },
    ]
)

best_parameters.to_csv(
    RESULTS_DIR
    / "best_hyperparameters.csv",
    index=False,
)

display(best_parameters)

,model,best_parameters,best_cv_macro_f1
0,Logistic Regression,"{'classifier__C': 0.5, 'classifier__class_weig...",0.528186
1,Linear SVM,"{'classifier__C': 0.05, 'classifier__class_wei...",0.522708


Grid search sees only the training split and keeps each entity and tweet-ID group within one fold. The supplied test split is not used until the final model has been selected.